In [2]:
import pandas as pd
import csv

DATA_PATH = "../data/reviews_nlp_rapidminer.csv"

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    quotechar='"',
    engine="python",
    encoding="utf-8"
)

print("Filas:", len(df))
print("Columnas:", df.columns.tolist())

# Limpiar posibles comillas en nombres de columnas
df.columns = df.columns.str.replace('"', '', regex=False).str.strip()

print("\nColumnas limpias:", df.columns.tolist())

print("\nDistribución sentiment_label:")
print(df["sentiment_label"].value_counts())

df.head()

Filas: 38931
Columnas: ['review_id', 'order_id', 'customer_unique_id', 'customer_city', 'customer_state', 'review_score', 'sentiment_label', 'clean_text', 'text_length']

Columnas limpias: ['review_id', 'order_id', 'customer_unique_id', 'customer_city', 'customer_state', 'review_score', 'sentiment_label', 'clean_text', 'text_length']

Distribución sentiment_label:
sentiment_label
positive    26222
negative     9272
neutral      3437
Name: count, dtype: int64


,review_id,order_id,customer_unique_id,customer_city,customer_state,review_score,sentiment_label,clean_text,text_length
0,97ca439bc427b48bc1cd7177abe71365,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,5,positive,perfeito produto entregue antes do combinado,45
1,06f45fcd8b9b54c30b0de110eb849228,000e63d38ae8c00bbcb5a30573b99628,860fc00d3154ce2346c43ebe47b9b6ce,sao paulo,SP,3,neutral,5,1
2,e79b1300216401c0212ab9b5a66372ff,00119ff934e539cf26f92b9ef0cdfed8,13df7b623839b4edc579ee40279d57c8,nova iguacu,RJ,5,positive,recebi o long 10 dias após a compra e chegou ...,173
3,28e20f3ef22e8795ea14e65f54e087a3,00169e31ef4b29deaae414f9a5e95929,577edb526f98771b20d6db4a51d79423,rio de janeiro,RJ,1,negative,entrega muito demorada ainda não recebi o pro...,50
4,df246bd48fbc0a0c9a7d4979c9192eec,001daeb0eddc45b999bad0801ad9d273,dc5aa19797b8fb7a4929ad109fb25553,itapicuru,BA,5,positive,produto aparentemente ótimo chegou muito ante...,160


In [3]:
# Nos quedamos solo con positive y negative
df_bin = df[df["sentiment_label"].isin(["positive", "negative"])].copy()

# Asegurar texto como string y quitar vacíos
df_bin["clean_text"] = df_bin["clean_text"].fillna("").astype(str)
df_bin = df_bin[df_bin["clean_text"].str.strip() != ""]

print("Filas binarias:", len(df_bin))
print("\nDistribución binaria:")
print(df_bin["sentiment_label"].value_counts())

X = df_bin["clean_text"]
y = df_bin["sentiment_label"]

Filas binarias: 35494

Distribución binaria:
sentiment_label
positive    26222
negative     9272
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=1992,
    stratify=y
)

sentiment_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=1992
    ))
])

sentiment_pipeline.fit(X_train, y_train)

y_pred = sentiment_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=["positive", "negative"]))

Accuracy: 0.9241243309230913

Classification report:
              precision    recall  f1-score   support

    negative       0.80      0.94      0.87      2782
    positive       0.98      0.92      0.95      7867

    accuracy                           0.92     10649
   macro avg       0.89      0.93      0.91     10649
weighted avg       0.93      0.92      0.93     10649


Confusion matrix:
[[7217  650]
 [ 158 2624]]


In [5]:
import joblib
import os

MODEL_PATH = "../models/sentiment_pipeline.pkl"

os.makedirs("../models", exist_ok=True)

joblib.dump(sentiment_pipeline, MODEL_PATH)

print(f"Modelo guardado en: {MODEL_PATH}")

Modelo guardado en: ../models/sentiment_pipeline.pkl


In [6]:
loaded_model = joblib.load("../models/sentiment_pipeline.pkl")

test_reviews = [
    "produto chegou antes do prazo e em perfeito estado",
    "produto chegou atrasado e veio quebrado",
    "péssima experiência, não recomendo",
    "excelente produto, gostei muito"
]

predictions = loaded_model.predict(test_reviews)
probabilities = loaded_model.predict_proba(test_reviews)

for text, pred, proba in zip(test_reviews, predictions, probabilities):
    print("Texto:", text)
    print("Predicción:", pred)
    print("Probabilidades:", dict(zip(loaded_model.classes_, proba)))
    print("-" * 60)

Texto: produto chegou antes do prazo e em perfeito estado
Predicción: positive
Probabilidades: {'negative': np.float64(0.005683563076448683), 'positive': np.float64(0.9943164369235513)}
------------------------------------------------------------
Texto: produto chegou atrasado e veio quebrado
Predicción: negative
Probabilidades: {'negative': np.float64(0.9113496081627873), 'positive': np.float64(0.08865039183721272)}
------------------------------------------------------------
Texto: péssima experiência, não recomendo
Predicción: negative
Probabilidades: {'negative': np.float64(0.9658519202140377), 'positive': np.float64(0.03414807978596237)}
------------------------------------------------------------
Texto: excelente produto, gostei muito
Predicción: positive
Probabilidades: {'negative': np.float64(0.012462261772263505), 'positive': np.float64(0.9875377382277365)}
------------------------------------------------------------


In [7]:
import json
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

metrics = {
    "model": "TF-IDF + Logistic Regression",
    "problem_type": "Binary sentiment classification",
    "classes": ["positive", "negative"],
    "accuracy": accuracy_score(y_test, y_pred),
    "classification_report": classification_report(y_test, y_pred, output_dict=True),
    "confusion_matrix_labels": ["positive", "negative"],
    "confusion_matrix": confusion_matrix(y_test, y_pred, labels=["positive", "negative"]).tolist()
}

os.makedirs("../outputs/metrics", exist_ok=True)

with open("../outputs/metrics/sentiment_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4, ensure_ascii=False)

print("Métricas guardadas en ../outputs/metrics/sentiment_metrics.json")

Métricas guardadas en ../outputs/metrics/sentiment_metrics.json


In [8]:
test_results = pd.DataFrame({
    "clean_text": X_test.values,
    "true_label": y_test.values,
    "predicted_label": y_pred
})

probas = sentiment_pipeline.predict_proba(X_test)
classes = sentiment_pipeline.classes_

for i, cls in enumerate(classes):
    test_results[f"prob_{cls}"] = probas[:, i]

os.makedirs("../outputs/metrics", exist_ok=True)

test_results.to_csv(
    "../outputs/metrics/sentiment_test_predictions.csv",
    index=False,
    encoding="utf-8"
)

print("Predicciones guardadas en ../outputs/metrics/sentiment_test_predictions.csv")
test_results.head()

Predicciones guardadas en ../outputs/metrics/sentiment_test_predictions.csv


,clean_text,true_label,predicted_label,prob_negative,prob_positive
0,meu produto quase três semanas antes do prazo ...,positive,positive,0.111317,0.888683
1,produto entregue dentro do prazo,positive,positive,0.080431,0.919569
2,rapidão antes do tempo produto bom,positive,positive,0.032074,0.967926
3,excelente,positive,positive,0.007894,0.992106
4,muito bom adorei,positive,positive,0.016295,0.983705
